# LeafDoctor: generate arm C images on Kaggle

Thin wrapper around `src/gen/diffusion_inpaint.py`; all logic lives in the repo. Stable Diffusion 1.5
inpainting paints a realistic scene around each arm B leaf placement, then the original leaf pixels are
pasted back, so disease symptoms are never changed.

Model: `stable-diffusion-v1-5/stable-diffusion-inpainting` (pinned revision, CreativeML OpenRAIL-M licence,
free and not gated), fp16, 25 steps, 512x512.

Before running:
1. **Add Data** → attach the private `ishaanbanga/leafdoctor-data` dataset.
   It is mounted read-only at `/kaggle/input/datasets/ishaanbanga/leafdoctor-data/`.
2. **Settings** → Accelerator: **GPU T4 x2**, Internet: on (to clone the repo and download the model, about 2 GB).
3. For a quick test, set `LIMIT_PER_CLASS = 2` (54 images) and check the timing and the preview grid.
   Then set it to 0 for all 2,700 images and use *Save Version* to run in the background.

Outputs: `/kaggle/working/synthetic/diffusion/<class_id>/*.jpg` with `diffusion_manifest.csv` (prompt, seed,
model per image), `/kaggle/working/synthetic/diffusion.zip` and `/kaggle/working/diffusion_preview.png`.
If the run stops, re-run the generation cell: images that already exist are skipped.

In [ ]:
# Parameters
REPO_URL = "https://github.com/bangaishaan/leafdoctor.git"
BRANCH = "main"
DATA_ROOT = "/kaggle/input/datasets/ishaanbanga/leafdoctor-data"   # read-only
OUT_DIR = "/kaggle/working/synthetic/diffusion"                   # images + diffusion_manifest.csv
PREVIEW = "/kaggle/working/diffusion_preview.png"
LIMIT_PER_CLASS = 0     # 0 = all 100 per class; e.g. 2 for a quick test
STEPS = 25
BATCH_SIZE = 4          # images per GPU per step

In [ ]:
%cd /kaggle/working
!rm -rf leafdoctor && git clone --depth 1 --branch {BRANCH} {REPO_URL} leafdoctor
%cd /kaggle/working/leafdoctor
!git log -1 --oneline
!pip install -q -r requirements.txt -r requirements-gen.txt

In [ ]:
import os
for name in ("plantvillage", "class_map.csv"):
    assert os.path.exists(os.path.join(DATA_ROOT, name)), f"{name} not found in {DATA_ROOT}; attach the dataset (Add Data)"
import torch
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 1, "no GPU: set Accelerator to GPU T4 x2"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# Generates on every visible GPU (one worker process each), then writes diffusion_manifest.csv,
# zips the outputs, prints the manifest summary and makes the 6x6 preview grid.
!python src/gen/diffusion_inpaint.py --data-root "{DATA_ROOT}" --out "{OUT_DIR}" --preview "{PREVIEW}" --steps {STEPS} --batch-size {BATCH_SIZE} --limit-per-class {LIMIT_PER_CLASS}

In [ ]:
# Delete the cloned repo so Kaggle has less to save, then show the preview.
import shutil
from IPython.display import Image as ShowImage, display

os.chdir("/kaggle/working")
shutil.rmtree("/kaggle/working/leafdoctor", ignore_errors=True)
print("deleted /kaggle/working/leafdoctor")
if os.path.exists(PREVIEW):
    display(ShowImage(PREVIEW))
!du -sh /kaggle/working/synthetic